<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Meet the simulator

**[Meet the simulator](https://learning.nextia-ai.com/courses/experiments/m01/meet-the-simulator/)** · Experiments and Causal Impact: Did Our Change Work? · Module 1, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read a test made by Larkfield's simulator, compute a difference and its 95% interval step by step, check it against the true effect, and see how far a naive comparison lands from the truth.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | A difference between two groups, a standard error and a 95% confidence interval, from the [mathematics course](https://learning.nextia-ai.com/courses/math/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m01/meet-the-simulator/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M01-L04-meet-the-simulator/meet-the-simulator-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 5 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "email/customers.csv.gz": "e16d24a15f1260876a6d73117b24604495f4af0abc855978b7e5b2828ecd3d3c",
    "email/truth.json": "80bc73295fefa15bf0567269825b5e71402abdc98180deaa302e3a05396b9fd3",
    "email/scenario.json": "e303618c4e8495c0b1627cbbe8f0c60269e8ccf706601749bc20d308219d49c9",
    "logs/customers.csv.gz": "0e9bd54ac28ef98bf94f6d1d0abe1bf0f8e51f67384d7b08c94b0b9310ab320a",
    "logs/truth.json": "8e2cd0eda8b31c1fba87687dfaf14b598d24921cf38734e8d18a21d8126b009f"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The reminder email test

Rafael sent a reminder email to customers who had not ordered for 30 days. A coin flip chose who got it: **B** got the email, **A** did not. The simulator wrote one row per customer. `ordered` is 1 if the customer ordered in the 14 days after the email.

> **Predict.** About 9% of these customers order in two weeks without an email. How many more percentage points do you expect with it?

In [ ]:
email = pd.read_csv("data/email/customers.csv.gz")
print(email.shape)
email.groupby("variant")["ordered"].agg(["count", "mean"])

> **Check.** You should see 60,000 rows; group A 30,208 customers with a mean of 0.0896, group B 29,792 customers with a mean of 0.1043.

## 2. The difference and its 95% interval, step by step

The difference is B − A. Its **standard error** says how much the difference would move if we ran the test again with other customers. For two proportions: SE = √(pA(1 − pA)/nA + pB(1 − pB)/nB). The 95% interval is the difference ± 1.96 × SE.

In [ ]:
n = email.groupby("variant")["ordered"].count()
p = email.groupby("variant")["ordered"].mean()
diff = p["B"] - p["A"]
se = np.sqrt(p["A"] * (1 - p["A"]) / n["A"] + p["B"] * (1 - p["B"]) / n["B"])
low, high = diff - 1.96 * se, diff + 1.96 * se
print(f"difference {diff:+.4f}, standard error {se:.4f}, 95% interval {low:+.4f} to {high:+.4f}")

> **Check.** You should see `difference +0.0147, standard error 0.0024, 95% interval +0.0100 to +0.0194`.

The email raised the share of customers who ordered by about 1.5 percentage points, and the interval does not include 0.

## 3. Check it against the truth

A real test never knows the true effect. The simulator does: it simulated every customer twice, with and without the email, with the same random numbers. `truth("email")` reads what it knows.

In [ ]:
t = truth("email")["ordered"]
print(f"true effect {t['effect']:+.4f} ({t['unit']})")
print("inside the interval:", low <= t["effect"] <= high)

> **Check.** You should see `true effect +0.0119` and `inside the interval: True`.

Our estimate is not the truth: it is one draw of chance. A 95% interval contains the true effect in about 95 tests out of 100. The course measured this: over 200 seeds of this scenario, the interval contained the truth in 97% of them.

## 4. A naive answer: users against non-users

The recommendation block went live for everyone. Nobody flipped a coin: customers chose to click it. Omar compares the customers who used the block with the others.

> **Predict.** The block's true effect is a few dollars per customer. How much more do you think its users spent?

In [ ]:
logs = pd.read_csv("data/logs/customers.csv.gz")
logs.groupby("used_block")["revenue"].agg(["count", "mean"])

> **Check.** You should see 13,338 users with a mean revenue of 39.19 dollars, and 26,662 non-users with 29.48 dollars.

## 5. Your turn: how much more did the users spend?

> **Your turn.** Compute how much more, in percent, the users spent than the non-users: (users' mean ÷ non-users' mean − 1) × 100. Put the number in `naive_pct`, rounded to 1 decimal. Then run the check cell.

In [ ]:
naive_pct = ...  # (users' mean revenue / non-users' mean revenue - 1) * 100, rounded to 1 decimal

In [ ]:
expect('naive_pct', naive_pct, 32.9)

In [ ]:
tl = truth("logs")
print(f"naive difference {tl['naive']['difference']:+.2f} dollars; true effect on the users {tl['att']['effect']:+.2f} dollars")

> **Check.** You should see `naive difference +9.70 dollars; true effect on the users +2.73 dollars`.

The users spent about a third more, but the block caused only a small part of it. The users were more active customers before they ever saw the block. This is the problem of the whole course: a comparison is an answer only when the groups were alike before the change.

## 6. The same in the course project

The course project does these steps with two commands. In a terminal, in your copy of `snapshots/start` (its README has the setup):

```sh
sim run --scenario email      # writes runs/email-s40/
xp analyse                    # the difference, its 95% interval and p-value
xp check                      # compared with the true effect
sim run --scenario logs
xp naive                      # users against non-users
xp check
xp reset                      # delete the runs and start again
```

The numbers are the same as in this notebook, because the seed is the same.

## Next

You can now run the simulator and check an answer against the truth. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m01/meet-the-simulator/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Randomise](https://learning.nextia-ai.com/courses/experiments/m02/randomise/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m01/meet-the-simulator/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.